# AirSense-R: Do Air-Quality Forecasters Keep Their Rank Under Sensor Degradation and Site Shift?

## 08 Post-results target-history checkpoint

**Permitted partitions: TRAIN/VALIDATION — REPLAY ONLY; FINAL TEST — FROZEN-MODEL INFERENCE.**

This is an explicitly post-results extension, separate from accepted P5–P8. Use a CPU runtime and **Run all**. Only Drive/GitHub authorization is needed; no SHA or stage entry. Requires the original P5 native models on Drive and the AIRSENSE_GITHUB_TOKEN Colab secret. Replays validation and accepted clean predictions before PM2.5 dropout (10/30/50%) and outages (6/12 hours). All models share each corrupted stream and primary causal imputation; future labels stay original. No training, new model selection, drift or noise. Original primary results remain unchanged.

Expect hours rather than minutes; duration depends on CPU and Drive. New predictions save after each model/seed to a separate Drive directory; compact evidence pushes to results/p9-target-history-01. One runtime only. After a network disconnect rerun cell 3; after runtime replacement Run all. Never delete a lock. Only confirmed saved checkpoints survive runtime loss. Execution completion is followed by independent evidence review. Low regret does not establish acceptable absolute forecast accuracy. See research/P9_CHECKPOINT_PLAN.md for the pre-execution extension plan.

## 1. Clone and verify source

In [1]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY = "https://github.com/Engr-Daniel/airsense-r.git"
# Use the full commit SHA of the GitHub revision containing this notebook.
# Resume using the SAME revision and run ID as the original execution.
TEST_MODE = os.environ.get("AIRSENSE_NOTEBOOK_TEST_MODE") == "1"
if TEST_MODE:
    ROOT = Path(os.environ["AIRSENSE_NOTEBOOK_TEST_ROOT"]).resolve()
else:
    CODE_REVISION = "0cd6c470471625409affb97dffbc4b3473663602"
    if len(CODE_REVISION) != 40 or any(c not in "0123456789abcdef" for c in CODE_REVISION.lower()):
        raise ValueError("Use a full Git commit SHA, not a moving branch name")
    ROOT = Path.cwd() / "airsense-r-p9-0cd6c4704716"
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(ROOT)], check=True)
        subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", CODE_REVISION], check=True)
    else:
        remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
        head = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
        problems = []
        if remote != REPOSITORY:
            problems.append("origin is not the expected repository")
        if head != CODE_REVISION:
            problems.append(f"checkout is {head}, requested {CODE_REVISION}")
        if dirty:
            problems.append("checkout has modified/untracked files")
        if problems:
            raise RuntimeError("; ".join(problems) + ". Preserve existing work and use a fresh runtime filesystem; restarting only the kernel does not remove the checkout.")
    if sys.version_info < (3, 11):
        raise RuntimeError("Python 3.11+ required")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[train,notebooks]"], check=True)
    print("Code revision:", CODE_REVISION)
# Editable installs may not activate their .pth file in an already-running kernel.
# Activate only this checkout, and reject cached modules from another checkout.
import importlib
SOURCE_DIR = (ROOT / "src").resolve()
PACKAGE_DIR = SOURCE_DIR / "airsense_r"
if not (PACKAGE_DIR / "__init__.py").is_file():
    raise RuntimeError("Selected checkout has no airsense_r package")
for module_name, module in tuple(sys.modules.items()):
    if module_name == "airsense_r" or module_name.startswith("airsense_r."):
        location = getattr(module, "__file__", None)
        if location is None or not Path(location).resolve().is_relative_to(PACKAGE_DIR):
            raise RuntimeError("Another airsense_r checkout is already imported; restart the runtime")
source_path = str(SOURCE_DIR)
if source_path in sys.path:
    sys.path.remove(source_path)
sys.path.insert(0, source_path)
importlib.invalidate_caches()
import airsense_r
if Path(airsense_r.__file__).resolve() != PACKAGE_DIR / "__init__.py":
    raise RuntimeError("Imported package does not match the selected checkout")
print("Workspace:", ROOT)
print("Verified package:", airsense_r.__file__)

from airsense_r.artifacts import source_hash
if source_hash(ROOT) != "dadc9f741c40c249573f9fb740f9d1d0275c89944594e6d89f06ed990c80b035":
    raise RuntimeError("Notebook/source mismatch; reopen the pinned notebook")


Code revision: 0cd6c470471625409affb97dffbc4b3473663602
Workspace: /content/airsense-r-p9-0cd6c4704716
Verified package: /content/airsense-r-p9-0cd6c4704716/src/airsense_r/__init__.py


## 2. Connect storage

In [2]:
from airsense_r.colab import configure_github_secret
from airsense_r.target_history import RUN_ID
if TEST_MODE:
    MODEL_DIRECTORY = Path(os.environ["AIRSENSE_P9_TEST_MODELS"])
    RUN_ID = os.environ["AIRSENSE_P9_TEST_RUN"]
    REMOTE = "fixture"
else:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    MODEL_DIRECTORY = Path("/content/drive/MyDrive/AirSense-R/model-checkpoints")
    configure_github_secret(ROOT, userdata.get("AIRSENSE_GITHUB_TOKEN"))
    REMOTE = REPOSITORY
if not (MODEL_DIRECTORY / "storage.json").is_file():
    raise FileNotFoundError("Original model storage missing; check the Drive account")
JOB_DIRECTORY = ROOT / ".tmp/notebook-jobs" / RUN_ID
print("Ready: frozen-model replay, then five PM2.5-history conditions. No training.")


Mounted at /content/drive
Ready: frozen-model replay, then five PM2.5-history conditions. No training.


## 3. Replay, evaluate and publish incrementally

In [3]:
from airsense_r.notebook_job import run_and_watch
command = [sys.executable, str(ROOT / "scripts/run_target_history.py"),
           "--model-directory", str(MODEL_DIRECTORY), "--remote", REMOTE]
if TEST_MODE:
    import json
    command = json.loads(os.environ["AIRSENSE_P9_TEST_COMMAND"])
run_and_watch(command, JOB_DIRECTORY, ROOT)


RUNNING: attached to the job. Rerunning this cell reconnects to the same job.
RUNNING: worker still active; waiting for the next progress checkpoint.
2026-10-29 16:41:42.723468: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
RUNNING: worker still active; waiting for the next progress checkpoint.
RUNNING: worker still active; waiting for the next progress checkpoint.
RUNNING: worker still active; waiting for the next progress checkpoint.
RUNNING: worker still active; waiting for the next progress checkpoint.
RUNNING: worker still active; waiting for the next progress checkpoint.
RUNNING: worker still active; waiting for the next progress checkpoint.
INFO:airsense_r.checkpoints:Checkpoint remotely confirmed: 2af0e4abc9ac5ee3dcb133db15f6a446ffaa5dee
INFO:airsense_r.checkpoints:Checkpoint remotely confirmed: 797ed7948cea24e7622731cadeca0aa7c7e85017
RUNNING: worker still active; waiti

{'finished': 1793294706.9849875,
 'identity': 'cd3817ca01bf54ce01b015e978b1df9738c07208e15dccd15fae5e023c0d9428',
 'pid': 6726,
 'return_code': 0,
 'started': 1793292071.3710027,
 'state': 'complete'}

## 4. Inspect completion

In [4]:
import json
from airsense_r.artifacts import RunStore, source_hash
from airsense_r.notebook_job import status
if status(JOB_DIRECTORY)["state"] != "complete":
    raise RuntimeError("Job incomplete; rerun cell 3 to reconnect or read its error")
directory = ROOT / ".tmp/checkpoint-checkouts" / RUN_ID / "results" / RUN_ID
manifest = json.loads((directory / "manifest.json").read_bytes())
provenance = manifest["provenance"]
if provenance["source_hash"] != source_hash(ROOT) or provenance["parameters"]["run_id"] != RUN_ID:
    raise RuntimeError("Extension provenance mismatch")
store = RunStore(directory, provenance)
if not store.load_json("preflight-complete.json")["passed"] or not store.load_json("complete.json")["execution_complete"]:
    raise RuntimeError("Replay or execution record missing")
display(store.load_json("complete.json"))
display(store.load_json("dashboard-target-history.json"))
print("EXECUTION COMPLETE; scientific acceptance awaits review. Save this notebook with outputs.")


{'conditions': 5,
 'execution_complete': True,
 'independent_intervals_checked': True,
 'post_results_extension': True,
 'refitting': False,
 'scientifically_accepted': False,
 'selected_family': 'persistence'}

{'reports': [{'condition': {'id': 'pm25-dropout-10',
    'kind': 'random_dropout',
    'severity': 0.1},
   'policy': 'primary',
   'scopes': {'full': {'independent_interval_check': True,
     'kendall_tau_b': 0.6666666666666666,
     'metrics': [{'clean_mae': 15.95741437488415,
       'clean_r2': 0.688383280078758,
       'clean_rank': 4.0,
       'clean_rmse': 37.165978947654295,
       'mae': 18.459244210743524,
       'model': 'gru',
       'r2': 0.667528649469574,
       'rank_change': 0.0,
       'rmse': 40.591852766431,
       'rpd_percent': 15.678165504037283,
       'stress_rank': 4.0},
      {'clean_mae': 10.810205954093624,
       'clean_r2': 0.946263106288692,
       'clean_rank': 3.0,
       'clean_rmse': 18.938472995098575,
       'mae': 13.664130455804878,
       'model': 'mlp',
       'r2': 0.920336133941269,
       'rank_change': 0.0,
       'rmse': 23.06821983313321,
       'rpd_percent': 26.400278716526458,
       'stress_rank': 3.0},
      {'clean_mae': 10.279643397

EXECUTION COMPLETE; scientific acceptance awaits review. Save this notebook with outputs.
